In [2]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# ---------- 設定 ----------
json_path = "./testdistance_estimates.json"
output_dir = "./scene_spline_fixed"
os.makedirs(output_dir, exist_ok=True)

# ---------- ジャンプ補正 ----------
def suppress_jumps(data, threshold=10.0):
    corrected = data.copy()
    for i in range(1, len(corrected)):
        if abs(corrected[i] - corrected[i - 1]) > threshold:
            corrected[i] = corrected[i - 1]
    return corrected

# ---------- 安定区間抽出 ----------
def find_stable_segments(data, diff_threshold=3.0, min_length=10, min_value=15.0):
    diffs = np.abs(np.diff(data, prepend=data[0]))
    stable_mask = (diffs < diff_threshold) & (data > min_value)

    segments = []
    start = None
    for i, val in enumerate(stable_mask):
        if val:
            if start is None:
                start = i
        else:
            if start is not None and i - start >= min_length:
                segments.append((start, i - 1))
            start = None
    if start is not None and len(data) - start >= min_length:
        segments.append((start, len(data) - 1))
    return segments

# ---------- スプライン補完（安定区間のみに適用） ----------
def apply_spline_fit_partial(x_all, data, stable_segments):
    stable_x = []
    stable_y = []
    for start, end in stable_segments:
        stable_x.extend(np.arange(start, end + 1))
        stable_y.extend(data[start:end + 1])

    if len(stable_x) < 4:
        return data  # 安定区間が少なすぎる場合はそのまま返す

    # 重複除去 + ソート
    unique_pairs = list({x: y for x, y in zip(stable_x, stable_y)}.items())
    if len(unique_pairs) < 4:
        return data

    unique_pairs.sort()
    sorted_x, sorted_y = zip(*unique_pairs)
    sorted_x = np.array(sorted_x)
    sorted_y = np.array(sorted_y)

    # スプライン補間器
    spline = CubicSpline(sorted_x, sorted_y, bc_type='natural')

    # 補間結果を jump-corrected をベースに置き換え
    smoothed = data.copy()
    for i in range(len(data)):
        if sorted_x[0] <= i <= sorted_x[-1]:
            smoothed[i] = spline(i)
    return smoothed

# ---------- メイン処理 ----------
if not os.path.exists(json_path):
    print(f"❌ JSONファイルが見つかりません: {json_path}")
else:
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    for scene_id, frame_data in data.items():
        frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
        distances = np.array([frame_data[k] for k in frame_keys], dtype=float)

        jump_corrected = suppress_jumps(distances)
        segments = find_stable_segments(jump_corrected, diff_threshold=3.0, min_length=10, min_value=15.0)
        x_all = np.arange(len(distances))
        smoothed = apply_spline_fit_partial(x_all, jump_corrected, segments)

        plt.figure(figsize=(10, 6))
        plt.plot(distances, label="Original", alpha=0.4, marker='o', markersize=3)
        plt.plot(jump_corrected, label="Jump-corrected", linestyle='--', marker='x', markersize=3)
        plt.plot(smoothed, label="Spline Smoothed", linewidth=2, color="green")
        plt.title(f"Scene {scene_id} - Spline Fit with Anchors (Partial)")
        plt.xlabel("Frame Index")
        plt.ylabel("Distance (m)")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        save_path = os.path.join(output_dir, f"{scene_id}_spline_fixed.png")
        plt.savefig(save_path)
        plt.close()

    print("✅ 全シーンにスプライン補完（部分適用）を実施し、保存が完了しました。")


✅ 全シーンにスプライン補完（部分適用）を実施し、保存が完了しました。


In [6]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# ---------- 設定 ----------
json_path = "./testdistance_estimates.json"
scene_id = "211"
output_path = f"./scene211_spline_from_original.png"

# ---------- 欠損や異常値（例えば0や小さすぎる値）をNaNにする ----------
def mask_invalid_values(data, min_valid=10.0):
    masked = data.copy()
    masked[masked < min_valid] = np.nan
    return masked

# ---------- スプライン補間（NaN除去したOriginalベース） ----------
def apply_spline_to_original(data, min_valid=10.0):
    masked = mask_invalid_values(data, min_valid)
    x_all = np.arange(len(data))
    valid_idx = np.where(~np.isnan(masked))[0]
    valid_values = masked[valid_idx]

    if len(valid_idx) < 4:
        return data

    spline = CubicSpline(valid_idx, valid_values, bc_type='natural')
    smoothed = data.copy()
    for i in range(len(data)):
        if valid_idx[0] <= i <= valid_idx[-1]:
            smoothed[i] = spline(i)
    return smoothed

# ---------- 実行 ----------
with open(json_path, encoding="utf-8") as f:
    all_data = json.load(f)

frame_data = all_data[scene_id]
frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
distances = np.array([frame_data[k] for k in frame_keys], dtype=float)

# 距離データ（Original）をそのまま使って補間
smoothed = apply_spline_to_original(distances, min_valid=10.0)

# ---------- グラフ描画 ----------
plt.figure(figsize=(10, 6))
plt.plot(distances, label="Original", alpha=0.4, marker='o', markersize=3)
plt.plot(smoothed, label="Spline Smoothed (from Original)", linewidth=2, color="green")
plt.title(f"Scene {scene_id} - Spline Fit from Original")
plt.xlabel("Frame Index")
plt.ylabel("Distance (m)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(output_path)
plt.close()

print(f"✅ Scene {scene_id} に Original ベースでスプライン補間を適用しました: {output_path}")


✅ Scene 211 に Original ベースでスプライン補間を適用しました: ./scene211_spline_from_original.png


In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# ---------- 設定 ----------
json_path = "./testdistance_estimates.json"
scene_id = "211"
output_path = f"./scene211_linear_interpolated.png"

# ---------- 欠損や異常値を NaN にする ----------
def mask_invalid(data, min_valid=10.0, max_valid=100.0):
    masked = data.copy()
    masked[(masked < min_valid) | (masked > max_valid)] = np.nan
    return masked

# ---------- 線形補間（NaNを補完） ----------
def linear_interpolate(data):
    x = np.arange(len(data))
    valid = ~np.isnan(data)
    if valid.sum() < 2:
        return data
    return np.interp(x, x[valid], data[valid])

# ---------- 実行 ----------
with open(json_path, encoding="utf-8") as f:
    all_data = json.load(f)

frame_data = all_data[scene_id]
frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
distances = np.array([frame_data[k] for k in frame_keys], dtype=float)

# 異常値を NaN にして線形補間
masked = mask_invalid(distances, min_valid=10.0)
interpolated = linear_interpolate(masked)

# ---------- グラフ描画 ----------
plt.figure(figsize=(10, 6))
plt.plot(distances, label="Original", alpha=0.4, marker='o', markersize=3)
plt.plot(interpolated, label="Linear Interpolated", linewidth=2, color="green")
plt.title(f"Scene {scene_id} - Linear Interpolation on Valid Points")
plt.xlabel("Frame Index")
plt.ylabel("Distance (m)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(output_path)
plt.close()

print(f"✅ Scene {scene_id} に線形補間を適用して保存しました: {output_path}")


✅ Scene 211 に線形補間を適用して保存しました: ./scene211_linear_interpolated.png
